In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.metrics import r2_score, mean_squared_error
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
from collections import defaultdict

In [ ]:
# df = pd.read_parquet("../Dane/chembl_ml_dataset_04.parquet")
df = pd.read_parquet("../Dane/chembl_ml_dataset_04_CHEMBL2147.parquet")

df = df[df["standard_type"] == "IC50"]
df = df[df["pchembl_value"].notna()]
df = df[df["canonical_smiles"].notna()]
target = df["target_chembl_id"].value_counts().idxmax()
df = df[df["target_chembl_id"] == target]
df = df.drop_duplicates("canonical_smiles")
df = df.reset_index(drop=True)

print("Target:", target)
print("Samples:", len(df))

## Featuryzacja RDKit

In [ ]:
# --- Atom feature helpers ---

ATOM_LIST = [
    'C', 'N', 'O', 'S', 'F', 'Si', 'P', 'Cl', 'Br', 'Mg', 'Na', 'Ca',
    'Fe', 'As', 'Al', 'I', 'B', 'V', 'K', 'Tl', 'Yb', 'Sb', 'Sn',
    'Ag', 'Pd', 'Co', 'Se', 'Ti', 'Zn', 'H', 'Li', 'Ge', 'Cu', 'Au',
    'Ni', 'Cd', 'In', 'Mn', 'Zr', 'Cr', 'Pt', 'Hg', 'Pb'
]
CHIRALITY_LIST = [
    Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
    Chem.rdchem.ChiralType.CHI_OTHER
]
HYBRIDIZATION_LIST = [
    Chem.rdchem.HybridizationType.S,
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
    Chem.rdchem.HybridizationType.OTHER
]

BOND_TYPE_LIST = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC
]
BOND_STEREO_LIST = [
    Chem.rdchem.BondStereo.STEREONONE,
    Chem.rdchem.BondStereo.STEREOANY,
    Chem.rdchem.BondStereo.STEREOZ,
    Chem.rdchem.BondStereo.STEREOE,
    Chem.rdchem.BondStereo.STEREOCIS,
    Chem.rdchem.BondStereo.STEREOTRANS
]


def one_hot(value, choices):
    """One-hot encoding z 'other' jako ostatnim bitem."""
    enc = [0] * (len(choices) + 1)
    try:
        idx = choices.index(value)
    except ValueError:
        idx = len(choices)  # 'other'
    enc[idx] = 1
    return enc


def atom_features(atom):
    """
    Wektor cech atomu (119 dim):
      - one-hot symbol atomu          : 44+1  = 45
      - one-hot chiralność            : 4+1   =  5 (nie ma 'other' w praktyce, ale bezpieczniej)
      - degree (liczba sąsiadów)      : 0-10  -> one-hot 11
      - formalne ładowanie            : one-hot [-5,-4,-3,-2,-1,0,1,2,3,4,5] -> 11
      - liczba H (total)              : 0-8   -> one-hot 9
      - hybridyzacja                  : 7+1   =  8
      - aromatyczność                 : 1 bit
      - czy w pierścieniu             : 1 bit
    Łącznie: 45+5+11+11+9+8+1+1 = 91 dim
    """
    feats = []
    # symbol
    feats += one_hot(atom.GetSymbol(), ATOM_LIST)
    # chiralność
    feats += one_hot(atom.GetChiralTag(), CHIRALITY_LIST)
    # degree (0-10)
    feats += one_hot(atom.GetDegree(), list(range(11)))
    # formal charge (-5..5)
    feats += one_hot(atom.GetFormalCharge(), list(range(-5, 6)))
    # total H
    feats += one_hot(atom.GetTotalNumHs(), list(range(9)))
    # hybridyzacja
    feats += one_hot(atom.GetHybridization(), HYBRIDIZATION_LIST)
    # aromatyczność
    feats.append(int(atom.GetIsAromatic()))
    # pierścień
    feats.append(int(atom.IsInRing()))
    return feats


def bond_features(bond):
    """
    Wektor cech wiązania (13 dim):
      - one-hot typ wiązania  : 4+1  = 5
      - one-hot stereo        : 6+1  = 7 (rzadko needed, ale kompletność)
      - czy w pierścieniu     : 1 bit
    Łącznie: 5+7+1 = 13 dim
    """
    feats = []
    feats += one_hot(bond.GetBondType(), BOND_TYPE_LIST)
    feats += one_hot(bond.GetStereo(), BOND_STEREO_LIST)
    feats.append(int(bond.IsInRing()))
    return feats


ATOM_DIM = len(atom_features(Chem.MolFromSmiles('C').GetAtomWithIdx(0)))
BOND_DIM = len(bond_features(Chem.MolFromSmiles('CC').GetBondWithIdx(0)))
print(f"Atom feature dim : {ATOM_DIM}")
print(f"Bond feature dim : {BOND_DIM}")

In [ ]:
def smiles_to_graph(smiles, y):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Node features
    x = torch.tensor(
        [atom_features(atom) for atom in mol.GetAtoms()],
        dtype=torch.float
    )

    # Edge index + edge attributes (obie kierunki)
    edge_index = []
    edge_attr  = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr  += [bf, bf]

    if len(edge_index) == 0:
        # isolated atom (np. single-atom SMILES)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(edge_attr,  dtype=torch.float)

    y_tensor = torch.tensor([y], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor)

## Dataset

In [ ]:
class MoleculeDataset(Dataset):
    def __init__(self, df):
        super().__init__()
        self.graphs = []
        self.smiles = []
        for _, row in df.iterrows():
            graph = smiles_to_graph(row["canonical_smiles"], row["pchembl_value"])
            if graph is not None:
                self.graphs.append(graph)
                self.smiles.append(row["canonical_smiles"])

    def len(self):
        return len(self.graphs)

    def get(self, idx):
        return self.graphs[idx]

## Scaffold Split

Scaffold split grupuje cząsteczki według ich rdzenia Murcko (wspólny szkielet pierścieniowy),
a następnie przydziela całe grupy do zbiorów train/test. Dzięki temu unikamy data leakage –
model nie widzi w treningu cząsteczek o identycznym szkielecie co testowe.

In [ ]:
def scaffold_split(dataset, frac_train=0.8, seed=42):
    """
    Implementacja Scaffold Split.
    Zwraca indeksy zbiorów train i test.
    """
    # 1. Wylicz scaffold dla każdej cząsteczki
    scaffold_to_indices = defaultdict(list)
    for idx, smi in enumerate(dataset.smiles):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            scaffold = ""
        else:
            try:
                scaffold = MurckoScaffold.MurckoScaffoldSmiles(
                    mol=mol, includeChirality=False
                )
            except Exception:
                scaffold = ""
        scaffold_to_indices[scaffold].append(idx)

    # 2. Posortuj grupy malejąco wg rozmiaru (większe scaffoldy → train)
    scaffolds = sorted(
        scaffold_to_indices.values(),
        key=lambda x: (len(x), x[0]),
        reverse=True
    )

    # 3. Rozdział na train / test
    n_total = len(dataset)
    train_cutoff = int(frac_train * n_total)

    train_idx, test_idx = [], []
    for group in scaffolds:
        if len(train_idx) < train_cutoff:
            train_idx.extend(group)
        else:
            test_idx.extend(group)

    print(f"Scaffold split: train={len(train_idx)}, test={len(test_idx)}")
    print(f"Unikalnych scaffoldów: {len(scaffold_to_indices)}")
    return train_idx, test_idx


dataset = MoleculeDataset(df)

train_idx, test_idx = scaffold_split(dataset, frac_train=0.8)

train_dataset = torch.utils.data.Subset(dataset, train_idx)
test_dataset  = torch.utils.data.Subset(dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

In [ ]:
def make_mlp(in_dim, hidden_dim, out_dim, dropout=0.0):
    """MLP używany wewnątrz warstwy GIN."""
    layers = [
        nn.Linear(in_dim, hidden_dim),
        nn.BatchNorm1d(hidden_dim),
        nn.ReLU(),
    ]
    if dropout > 0:
        layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(hidden_dim, out_dim))
    return nn.Sequential(*layers)


class GIN(nn.Module):
    def __init__(self, input_dim=ATOM_DIM, hidden_dim=256, num_layers=5, dropout=0.2):
        super().__init__()
        self.dropout = dropout

        # Warstwy GINConv
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        for i in range(num_layers):
            in_d  = input_dim if i == 0 else hidden_dim
            mlp   = make_mlp(in_d, hidden_dim, hidden_dim, dropout=0.0)
            self.convs.append(GINConv(mlp, train_eps=True))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        # Pooling: konkatenacja sum z każdej warstwy (JK-sum)
        self.pool = global_add_pool

        # MLP predykcyjny
        self.lin1 = nn.Linear(hidden_dim * num_layers, 128)
        self.lin2 = nn.Linear(128, 1)
        self.num_layers = num_layers

    def forward(self, x, edge_index, batch):
        layer_outs = []

        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            layer_outs.append(self.pool(x, batch))   # [batch, hidden]

        # Jumping Knowledge: łączymy reprezentacje ze wszystkich warstw
        out = torch.cat(layer_outs, dim=-1)           # [batch, hidden * num_layers]

        out = self.lin1(out)
        out = F.relu(out)
        out = F.dropout(out, p=self.dropout, training=self.training)
        out = self.lin2(out)
        return out

In [ ]:
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = GIN(input_dim=ATOM_DIM, hidden_dim=256, num_layers=5, dropout=0.2).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nLiczba parametrów: {n_params:,}")

## Trening

In [ ]:
EPOCHS = 301
train_losses = []
val_losses   = []

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out  = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y.view(-1, 1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    train_losses.append(train_loss)

    # --- Eval ---
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out  = model(batch.x, batch.edge_index, batch.batch)
            loss = criterion(out, batch.y.view(-1, 1))
            total_val_loss += loss.item()

    val_loss = total_val_loss / len(test_loader)
    val_losses.append(val_loss)

    if epoch % 50 == 0:
        print(f"Epoch {epoch:>4}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    if epoch % 100 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, f'../Dane/Modele/gin_checkpoint_{epoch}.pt')

In [ ]:
model.eval()
preds   = []
targets = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out   = model(batch.x, batch.edge_index, batch.batch)
        preds.extend(out.cpu().numpy().flatten())
        targets.extend(batch.y.cpu().numpy().flatten())

preds   = np.array(preds)
targets = np.array(targets)

r2   = r2_score(targets, preds)
rmse = np.sqrt(mean_squared_error(targets, preds))
print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")

## Wyniki

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Krzywa strat
axes[0].plot(train_losses, label="Train loss")
axes[0].plot(val_losses,   label="Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Training curves (GIN + Scaffold split)")
axes[0].legend()

# Preds vs targets
axes[1].scatter(targets, preds, alpha=0.4, s=15)
mn, mx = targets.min(), targets.max()
axes[1].plot([mn, mx], [mn, mx], 'r--', label='ideal')
axes[1].set_xlabel("True pChEMBL")
axes[1].set_ylabel("Predicted pChEMBL")
axes[1].set_title(f"R²={r2:.3f}  RMSE={rmse:.3f}")
axes[1].legend()

plt.tight_layout()
plt.show()

## Dotrenowanie z LR Scheduler

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=30
)

In [ ]:
START_EPOCH = 301
END_EPOCH   = 701

train_losses_cont = []
val_losses_cont   = []

for epoch in range(START_EPOCH, END_EPOCH):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out  = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y.view(-1, 1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    train_losses_cont.append(train_loss)

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out  = model(batch.x, batch.edge_index, batch.batch)
            loss = criterion(out, batch.y.view(-1, 1))
            total_val_loss += loss.item()

    val_loss = total_val_loss / len(test_loader)
    val_losses_cont.append(val_loss)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch:>5}: "
            f"train_loss={train_loss:.4f}, "
            f"val_loss={val_loss:.4f}, "
            f"lr={current_lr:.6f}"
        )

    if epoch % 100 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, f'../Dane/Modele/gin_checkpoint_{epoch}.pt')

In [ ]:
model.eval()
preds   = []
targets = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out   = model(batch.x, batch.edge_index, batch.batch)
        preds.extend(out.cpu().numpy().flatten())
        targets.extend(batch.y.cpu().numpy().flatten())

preds   = np.array(preds)
targets = np.array(targets)

r2   = r2_score(targets, preds)
rmse = np.sqrt(mean_squared_error(targets, preds))
print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")

In [ ]:
epoki = range(START_EPOCH, START_EPOCH + len(train_losses_cont))
plt.figure(figsize=(10, 5))
plt.plot(epoki, train_losses_cont, label="Train loss")
plt.plot(epoki, val_losses_cont,   label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training continuation (GIN)")
plt.legend()
plt.show()

In [ ]:
torch.save(model.state_dict(), "../Dane/gin_01.pt")